# Binary GP Classification と Active Learning

2値分類で、latent GP と class probability を分離して確認し、分類専用 acquisition から次にラベル付けする候補を生成します。

## このNotebookで確認すること

- Bernoulli likelihood を持つ variational GP classifier の構築と学習
- latent posterior と class probability の違い
- latent / probability / label uncertainty の違い
- Predictive Entropy と BALD による Active Learning
- BoTorch `optimize_acqf` による q=1 candidate generation

関連理論: [Gaussian Process Classification](../../docs/theory/24_classification.md) / [Classification Active Learning](../../docs/theory/acquisition/classification-active-learning.md)

```text
Data -> Model -> Variational fit -> Latent posterior -> Class probability
     -> Classification acquisition -> optimize_acqf -> Candidate -> Label evaluation
```

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.optim import optimize_acqf

from robotorchan.acquisition import BALD, PredictiveEntropy
from robotorchan.models.classification import BinarySingleTaskGPClassifier

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Binary classification problem

潜在境界が分かる1次元の合成問題を使います。観測ラベルは0/1です。

In [ ]:
def true_label(X: torch.Tensor) -> torch.Tensor:
    return (X.squeeze(-1) > 0.5).to(dtype=torch.double)


train_X = torch.tensor([[0.05], [0.15], [0.30], [0.70], [0.85], [0.95]])
train_Y = true_label(train_X)
bounds = torch.tensor([[0.0], [1.0]])

print(train_X.shape, train_Y.shape)
print(train_Y)

## 3. Model construction

`posterior(X)` はclass probabilityではなくlatent function posteriorです。

In [ ]:
model = BinarySingleTaskGPClassifier(
    train_X,
    train_Y,
    inducing_points=6,
)

print(model.classification_metadata)
print(model.raw_train_X.shape, model.raw_train_Y.shape)
print(type(model.make_mll()).__name__)

## 4. Variational fitting

Bernoulli likelihoodなのでExactMarginalLogLikelihoodではなくVariationalELBOを学習します。

In [ ]:
model.train()
model.likelihood.train()
mll = model.make_mll()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for _ in range(40):
    optimizer.zero_grad()
    output = model(model.train_inputs[0])
    loss = -mll(output, model.train_targets)
    loss.backward()
    optimizer.step()

model.eval()
model.likelihood.eval()
print(f"final loss = {loss.item():.4f}")

## 5. Latent posterior と class probability

latent meanは確率ではありません。`predict_proba`はlikelihoodを通してlatent uncertaintyを積分した予測確率です。

In [ ]:
grid = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
with torch.no_grad():
    latent = model.latent_posterior(grid)
    latent_mean = latent.mean.squeeze(-1)
    latent_std = latent.variance.clamp_min(0.0).sqrt().squeeze(-1)
    probability = model.predict_proba(grid)[..., 1]

print(latent_mean.shape, probability.shape)

## 6. 3種類の不確実性

`latent_variance`、class-probability samplesの分散、predictive entropyを別々に計算します。

In [ ]:
with torch.no_grad():
    latent_variance = model.latent_variance(grid).squeeze(-1)
    probability_samples = model.sample_class_probabilities(
        grid,
        sample_shape=torch.Size([64]),
    )
    probability_variance = probability_samples[..., 1].var(dim=0, unbiased=False)
    label_entropy = model.predictive_entropy(grid)

print(latent_variance.shape, probability_variance.shape, label_entropy.shape)

## 7. Classification Active Learning

Predictive Entropyはlabel ambiguity、BALDはそのうちlatent/model uncertaintyに由来する情報量を評価します。

In [ ]:
entropy_acq = PredictiveEntropy(model)
bald_acq = BALD(model, num_samples=64)

Xq = grid.unsqueeze(-2)
with torch.no_grad():
    entropy_score = entropy_acq(Xq)
    bald_score = bald_acq(Xq)

print(entropy_score.shape, bald_score.shape)

## 8. Candidate generation

分類ALは現在q=1です。robotorchanの分類acquisitionを標準BoTorch optimizerへそのまま渡します。

In [ ]:
candidate, acquisition_value = optimize_acqf(
    acq_function=entropy_acq,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
next_label = true_label(candidate.detach()).to(dtype=torch.long)

print("candidate:", candidate.detach().squeeze().item())
print("acquisition value:", acquisition_value.detach().item())
print("observed label:", next_label.item())

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].plot(grid.squeeze(-1), probability, label="P(y=1)")
axes[0].scatter(train_X.squeeze(-1), train_Y, label="observations")
axes[0].axvline(candidate.detach().squeeze().item(), linestyle="--", label="next candidate")
axes[0].set_ylabel("probability / label")
axes[0].legend()

axes[1].plot(grid.squeeze(-1), entropy_score, label="PredictiveEntropy")
axes[1].plot(grid.squeeze(-1), bald_score, label="BALD")
axes[1].set_xlabel("x")
axes[1].set_ylabel("acquisition")
axes[1].legend()
plt.show()

## 10. 使い所と注意点

- 通常の2値分類では `BinarySingleTaskGPClassifier` を入口にします。
- `posterior()` はlatent posteriorで、class probabilityではありません。
- Probability VarianceとBernoulli predictive varianceは別の量です。
- 回帰用qEIやStraddleを、Gaussian latent posteriorという理由だけで分類へ流用しません。
- 現在のclassification Active Learningはq=1です。
- Entropy / Margin / Probability Variance / BALDの契約はmulticlassへ拡張可能ですが、Latent Straddleはbinary固有です。

このNotebookは候補を1点生成して真のlabelを評価するところまでを示します。実運用では、その観測をtraining dataへ追加して再fitする処理を反復します。